In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load the cleaned dataset
df = pd.read_csv("cleaned_employee_dataset.csv")

# 2. Separate features and target variable
X = df.drop("Salary_INR", axis=1)
y = df["Salary_INR"]

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Selecting final features for deployment...")

# 4. Feature Selection
base_model = LinearRegression()
selector = SequentialFeatureSelector(base_model, n_features_to_select=8, direction="forward")
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = list(X.columns[selector.get_support()])
print(f"Final features selected: {selected_features}")

print("\nTraining the final Ridge Regression model...")

# 5. Train the final model
final_model = Ridge(alpha=1.0)
final_model.fit(X_train_selected, y_train)

# Evaluate to ensure it works
predictions = final_model.predict(X_test_selected)
r2 = r2_score(y_test, predictions)
print(f"Final Model R2 Score: {r2:.4f}")

# 6. Save the final deployment package
with open('final_salary_model.pkl', 'wb') as f:
    pickle.dump({
        'model': final_model,
        'selector': selector,
        'feature_names': selected_features
    }, f)

print("\nSystem ready. Model saved successfully as 'final_salary_model.pkl'.")

Selecting final features for deployment...
Final features selected: ['Age', 'Experience', 'JobRole', 'PerformanceRating', 'ProjectsCompleted', 'LeadershipScore', 'Promotions', 'LocationType']

Training the final Ridge Regression model...
Final Model R2 Score: 0.9597

System ready. Model saved successfully as 'final_salary_model.pkl'.
